Здесь я объединяю данные экспериментов, предоставленные научным руководителем, в один датасет, очищаю его от проблемных записей и кодирую классы.

In [9]:
import polars as pl
dataset_1 = pl.read_excel('../data/transcribed/object_naming.xlsx')
dataset_2 = pl.read_excel('../data/transcribed/'
                              'object&action_naming.xlsx')


Could not determine dtype for column 9, falling back to string
Could not determine dtype for column 12, falling back to string


In [39]:
columns_for_now = [
    'pID',
    'StimSite',
    'Stimulus',
    'RT_start',
    'Response_annot',
    'Response_transcription_annot',
    'Error_type_annot'
]

dataset = pl.concat([
    dataset_1[columns_for_now].with_columns(pl.lit(1).alias('exp')),
    dataset_2.rename({'RT_start_annot': 'RT_start'})[columns_for_now].with_columns(pl.lit(2).alias('exp'))
])


In [40]:
print(len(dataset))

22260


In [41]:
dataset = dataset\
    .with_columns(pl.col("Error_type_annot").str.to_lowercase())


In [42]:
pl.Config.set_tbl_rows(-1)
print(
*dataset['Error_type_annot']\
    .value_counts()\
    .sort('count', descending=True).rows(),
    sep='\n'
)


(None, 19807)
('задержка', 1151)
('дизартрия', 414)
('фонетическая парафазия', 337)
('семантическая парафазия', 185)
('аномия', 124)
('поиск слова', 78)
('speech arrest', 32)
('смешанная парафазия', 19)
('семантическая парафазия + задержка', 18)
('персеверация', 18)
('дизартрия + задержка', 13)
(' ', 10)
('поиск слова + задержка', 8)
('фонетическая парафазия + задержка', 6)
('семантическая парафазия + дизартрия', 5)
('задержка + дизартрия', 5)
('задержка + поиск слова', 4)
('s/a', 3)
('поиск слова + дизартрия', 2)
('дизартрия + семантическая парафазия + задержка', 1)
('поиск слова?', 1)
('фонетическая парфазия', 1)
('дизартрия + фонетическая парафазия', 1)
('задржка', 1)
('аномия + дизартрия', 1)
('задрежка', 1)
('поиск слова или семантическая парафазия (хотел сказать "жираф"?)', 1)
('смешанная парафазия ', 1)
('семантическая парафазия + фонетическая парафазия', 1)
('задрержка', 1)
('диазртрия + задержка', 1)
("тут брат… плач'ьт", 1)
('поиск слова (более вероятно) или дизартиря', 1)
('

In [43]:
dataset = dataset.with_columns(
    pl.col('Error_type_annot').replace({
    ' ': None,
    's/a': 'speech arrest',
    'фонетическа парафазия': 'фонетическая парафазия',
    'фонетическая парфазия': 'фонетическая парафазия',
    'задрежка': 'задержка',
    'задржка': 'задержка',
    'подбор слова': 'поиск слова'
    })
)


In [44]:
dataset[['Error_type_annot']] = dataset[['Error_type_annot']]\
                                                    .fill_null('нет')

In [45]:
allowed_types = dataset['Error_type_annot']\
        .value_counts()\
        .sort('count', descending=True)['Error_type_annot'][:8]


In [46]:
print(f'Изначальная длина: {len(dataset)}')

dataset_clean = dataset.filter(
    pl.col('Error_type_annot').is_in(allowed_types)
)

print(f'Без плохих аннотаций ошибок: {len(dataset_clean)}')

# ошибочные поинты
dataset_clean = dataset_clean.filter(
    ~((pl.col('Error_type_annot') == 'нет') & 
    (pl.col('Response_annot').is_null()))
)

print(f'Без плохих поинтов (без типа и текста): {len(dataset_clean)}')

# частные случаи неверной разметки S/A
dataset_clean = dataset_clean.with_columns(
    pl.when(pl.col('Response_annot').is_in({'сушит',
                                            'трёт',
                                            'это черепаха'
                                            }) &
            (pl.col('Error_type_annot') == 'speech arrest')
     ).then(pl.lit('нет'))
     .otherwise(pl.col('Error_type_annot'))
     .alias('Error_type_annot')
)


Изначальная длина: 22260
Без плохих аннотаций ошибок: 22146
Без плохих поинтов (без типа и текста): 22101


C:\Users\Vlad\AppData\Local\Temp\ipykernel_9848\2857271951.py:3: DeprecationWarning: `is_in` with a collection of the same datatype is ambiguous and deprecated.
Please use `implode` to return to previous behavior.

See https://github.com/pola-rs/polars/issues/22149 for more information.
  dataset_clean = dataset.filter(


In [ ]:
dataset_clean.head()


In [47]:
error_type_ids = {
    'нет': 0,
    'speech arrest': 1,
    'аномия': 2,
    'дизартрия': 3,
    'задержка': 4,
    'поиск слова': 5,
    'семантическая парафазия': 6,
    'фонетическая парафазия': 7
}


In [ ]:
from json import dump

error_type_ids_file = open('../data/processed/error_type_ids.json',
                           'w', -1, 'utf-8')
dump(error_type_ids, error_type_ids_file)
error_type_ids_file.close()


In [48]:
dataset_clean_label = dataset_clean.with_columns(
    pl.col('Error_type_annot')\
    .replace(error_type_ids)\
    .cast(pl.Int8)
    .alias('error_type_label')
)


In [ ]:
dataset_clean_label.sample(10)


Количество уникальных ошибок у испытуемых в одной зоне

In [62]:
dataset_clean_label\
    .filter(pl.col('error_type_label') > 0)\
    .group_by('exp', 'pID', 'StimSite')\
    .agg(pl.col('error_type_label').n_unique().alias('uniques'))\
    .sort('uniques', descending=True)\
    ['uniques']\
    .value_counts()
    

uniques,count
u32,u32
4,1
3,37
2,312
1,1241


In [24]:
dataset_clean_label.write_csv('../data/processed/'
                               'dataset_clean_label.csv')


In [49]:
print(len(dataset_clean_label))


22101
